In [46]:

import os
os.environ["TORCH_ONNX_FORCE_OLD_EXPORTER"] = "1"
from onnx2pytorch import ConvertModel
from torch import nn
import torch.nn.functional as F
import onnx
import idx2numpy, numpy as np, torch
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
import onnxruntime as ort

In [32]:
model_path = "D:\\automated-object-control\\lab-for-block2\\nn_model.onnx"
data_idx_path = "D:\\automated-object-control\\lab-for-block2\\trainingDatas.idx"
label_idx_path = "D:\\automated-object-control\\lab-for-block2\\trainingLabels.idx"
onnx_model = onnx.load(model_path)
model = ConvertModel(onnx_model, experimental=True)
model.train()

ConvertModel(
  (MatMul_sequential/dense_1/BiasAdd:0): Linear(in_features=4, out_features=64, bias=True)
  (Relu_sequential/dense_1/Relu:0): ReLU(inplace=True)
  (MatMul_output_layer): Linear(in_features=64, out_features=3, bias=True)
)

In [33]:
X_idx = idx2numpy.convert_from_file(data_idx_path) #expect shape (N,4) or (N,1,4)
Y_idx = idx2numpy.convert_from_file(label_idx_path) #expect shape (N,)

print("X shape:", X_idx.shape, "| dtype:", X_idx.dtype)
print("Y shape:", Y_idx.shape, "| dtype:", Y_idx.dtype)

# 2) make contiguous writable copies with desired dtypes
X = torch.tensor(np.array(X_idx, dtype=np.float32, copy=True))
Y = torch.tensor(np.array(Y_idx, dtype=np.int64,  copy=True))

print("X:", X.shape, X.dtype, " | Y:", Y.shape, Y.dtype)

train_dataset = TensorDataset(X,Y)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=True)

X shape: (8006, 4) | dtype: >f4
Y shape: (8006,) | dtype: >i4
X: torch.Size([8006, 4]) torch.float32  | Y: torch.Size([8006]) torch.int64


In [34]:
criterion = nn.CrossEntropyLoss()
model.eval()
with torch.no_grad():
    logits = model(X[:256].float())
    ce = criterion(logits, Y[:256].long())
    pred = logits.argmax(1)
    acc = (pred == Y[:256]).float().mean().item()
print("CE(loss) first 256:", ce.item(), " | acc:", round(acc,4))

CE(loss) first 256: 0.2351202815771103  | acc: 0.9102


In [35]:
model.train()
criterion = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)

def robustness_property_loss(model, x, y, eps=0.01, steps=3, alpha=None):
    if alpha is None: alpha = eps/2
    x_adv = x.detach().clone().requires_grad_(True)
    for _ in range(steps):
        logits = model(x_adv)
        fy = logits.gather(1, y.view(-1,1))
        margins = logits - fy
        margins.scatter_(1, y.view(-1,1), float('-inf'))
        worst = margins.max(1).values.mean()
        grad, = torch.autograd.grad(worst, x_adv)
        with torch.no_grad():
            x_adv += alpha * torch.sign(grad)
            x_adv = torch.max(torch.min(x_adv, x + eps), x - eps)  # L∞ projection
            # clamp here to YOUR input range if needed (e.g., 0..1 or z-score bounds)
        x_adv.requires_grad_(True)
    logits = model(x_adv)
    fy = logits.gather(1, y.view(-1,1))
    margins = logits - fy
    margins.scatter_(1, y.view(-1,1), float('-inf'))
    return F.relu(margins.max(1).values).mean()

In [36]:
lambda_start, lambda_end = 0.1, 3.0
eps_start,    eps_end    = 0.003, 0.02
epochs = 10

for epoch in range(epochs):
    t = epoch / max(1, epochs-1)
    lam = lambda_start + (lambda_end - lambda_start)*t
    eps = eps_start + (eps_end - eps_start)*t
    for x, y in train_loader:        # ensure x has shape [B, 4] with correct scaling
        x, y = x.float(), y.long()
        logits = model(x)
        data_loss = criterion(logits, y)
        prop_loss = robustness_property_loss(model, x, y, eps=eps, steps=3)
        loss = data_loss + lam * prop_loss
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        print(f"epoch {epoch+1}/{epochs} loss={loss:.3f} λ={lam:.2f} ε={eps:.4f}")

epoch 1/10 loss=0.159 λ=0.10 ε=0.0030
epoch 1/10 loss=0.159 λ=0.10 ε=0.0030
epoch 1/10 loss=0.186 λ=0.10 ε=0.0030
epoch 1/10 loss=0.146 λ=0.10 ε=0.0030
epoch 1/10 loss=0.253 λ=0.10 ε=0.0030
epoch 1/10 loss=0.134 λ=0.10 ε=0.0030
epoch 1/10 loss=0.145 λ=0.10 ε=0.0030
epoch 1/10 loss=0.129 λ=0.10 ε=0.0030
epoch 1/10 loss=0.212 λ=0.10 ε=0.0030
epoch 1/10 loss=0.245 λ=0.10 ε=0.0030
epoch 1/10 loss=0.121 λ=0.10 ε=0.0030
epoch 1/10 loss=0.146 λ=0.10 ε=0.0030
epoch 1/10 loss=0.257 λ=0.10 ε=0.0030
epoch 1/10 loss=0.099 λ=0.10 ε=0.0030
epoch 1/10 loss=0.109 λ=0.10 ε=0.0030
epoch 1/10 loss=0.186 λ=0.10 ε=0.0030
epoch 1/10 loss=0.189 λ=0.10 ε=0.0030
epoch 1/10 loss=0.243 λ=0.10 ε=0.0030
epoch 1/10 loss=0.225 λ=0.10 ε=0.0030
epoch 1/10 loss=0.209 λ=0.10 ε=0.0030
epoch 1/10 loss=0.118 λ=0.10 ε=0.0030
epoch 1/10 loss=0.190 λ=0.10 ε=0.0030
epoch 1/10 loss=0.098 λ=0.10 ε=0.0030
epoch 1/10 loss=0.172 λ=0.10 ε=0.0030
epoch 1/10 loss=0.223 λ=0.10 ε=0.0030
epoch 1/10 loss=0.149 λ=0.10 ε=0.0030
epoch 1/10 l

In [37]:
model.eval()
with torch.no_grad():
    logits = model(X)
    pred = logits.argmax(1)
    acc = (pred == Y).float().mean().item()
print("train accuracy:", round(acc, 4))

train accuracy: 0.9633


In [38]:
# quick random 80/20 split check (repeat a couple of times)
perm = torch.randperm(len(X))
cut = int(0.8*len(X))
idx_tr, idx_te = perm[:cut], perm[cut:]
with torch.no_grad():
    acc_te = (model(X[idx_te]).argmax(1) == Y[idx_te]).float().mean().item()
print("held-out acc:", round(acc_te,4))

held-out acc: 0.97


In [42]:
save_path = Path("nn_model_finetuned.onnx")
save_path.parent.mkdir(parents=True, exist_ok=True)

model.eval()
torch.onnx.export(
    model,
    X[:1].float(),              # example input
    save_path.as_posix(),
    input_names=["input"],
    output_names=["logits"],
    opset_version=18,           # ONNX opset to use
    export_params=True,
    do_constant_folding=True,
)

C:\Users\Baoyan Si\miniconda3\envs\automated-object-control\lib\site-packages\torch\onnx\utils.py:1547: OnnxExporterWarning: Exporting to ONNX opset version 18 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 17. To use a newer opset version, consider 'torch.onnx.dynamo_export()'. Note that dynamo_export() is in preview. Please report errors with dynamo_export() as Github issues to https://github.com/pytorch/pytorch/issues.
  warnings.warn(


In [47]:

onnx_path = "nn_model_finetuned.onnx"

# ✅ 1) Check that the ONNX file is structurally valid
onnx.checker.check_model(onnx_path)
print("✅ Model structure verified")

# ✅ 2) Inspect I/O names and shapes
model = onnx.load(onnx_path)
print("Inputs:", [i.name for i in model.graph.input])
print("Outputs:", [o.name for o in model.graph.output])

# ✅ 3) Run an inference test with ONNX Runtime
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
example = np.random.randn(1, 4).astype(np.float32)
onnx_out = sess.run(None, {"input": example})[0]

print("ONNX output sample:", onnx_out)
print("✅ ONNX model runs successfully")

✅ Model structure verified
Inputs: ['input']
Outputs: ['logits']
ONNX output sample: [[-5.608925  -1.2469702  4.423918 ]]
✅ ONNX model runs successfully
